# Prototype: Retrieval Evaluation for RAG

Colab-ready prototype of the experiments from `01_retrieval_eval`.

Goal: compare sparse, dense and hybrid retrieval for a documentation QA scenario using the Wix/WixQA dataset from HuggingFace.

The notebook covers dataset loading, corpus filtering, chunking, BM25, dense retrieval, hybrid linear fusion, MRR/Recall/Hit metrics and top-k selection.

## 1. Install dependencies

In Colab, enable GPU via `Runtime -> Change runtime type -> T4 GPU` before running dense embedding cells.

In [ ]:
!pip -q install datasets sentence-transformers rank-bm25 pandas numpy tqdm

In [ ]:
import json
import math
import re
from collections import defaultdict
from dataclasses import dataclass
from typing import Any

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 2. Experiment configuration

`MAX_DOCS` and `MAX_QA` keep the notebook quick for a prototype. Set them to `None` for a fuller run.

In [ ]:
DATASET_NAME = "Wix/WixQA"
EXCLUDED_ARTICLE_TYPES = {"feature_request"}

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
# For stronger but slower experiments, try:
# EMBED_MODEL = "BAAI/bge-small-en-v1.5"

EVAL_KS = [1, 3, 5, 6, 8, 10, 15, 20]
ALPHAS = [round(x / 10, 1) for x in range(11)]
CHUNK_TARGET_CHARS = 600
CHUNK_OVERLAP_SENTENCES = 1

MAX_DOCS = None   # e.g. 2000 for a faster prototype
MAX_QA = None     # e.g. 100 for a faster prototype

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 3. Load Wix/WixQA from HuggingFace

The dataset format may contain several splits. The helper below searches for corpus-like rows and QA-like rows by column names.

In [ ]:
raw = load_dataset(DATASET_NAME)
raw

In [ ]:
def rows_from_dataset(ds) -> list[dict[str, Any]]:
    rows = []
    for split_name, split in ds.items():
        for row in split:
            item = dict(row)
            item["_split"] = split_name
            rows.append(item)
    return rows

def first_value(row: dict[str, Any], keys: list[str], default=None):
    for key in keys:
        if key in row and row[key] not in (None, ""):
            return row[key]
    return default

rows = rows_from_dataset(raw)
print("rows:", len(rows))
print("sample columns:", sorted(rows[0].keys()))
pd.DataFrame(rows[:3])

## 4. Normalize corpus and QA pairs

The production experiment filters `feature_request` documents and evaluates only single-article QA pairs.

In [ ]:
DOC_ID_KEYS = ["doc_id", "document_id", "article_id", "id", "page_id"]
TITLE_KEYS = ["title", "article_title", "page_title"]
TEXT_KEYS = ["contents", "content", "text", "article", "body"]
TYPE_KEYS = ["article_type", "type", "doc_type"]
QUESTION_KEYS = ["question", "query", "prompt"]
ANSWER_KEYS = ["answer", "response", "gold_answer"]
GOLD_KEYS = ["doc_id", "document_id", "article_id", "gold_doc_id", "gold_document_id", "page_id"]

def normalize_doc(row: dict[str, Any]) -> dict[str, Any] | None:
    doc_id = first_value(row, DOC_ID_KEYS)
    title = first_value(row, TITLE_KEYS, "")
    text = first_value(row, TEXT_KEYS, "")
    article_type = first_value(row, TYPE_KEYS, "article")
    if not doc_id or not text:
        return None
    if article_type in EXCLUDED_ARTICLE_TYPES:
        return None
    return {"id": str(doc_id), "title": str(title), "text": str(text), "article_type": str(article_type)}

def normalize_qa(row: dict[str, Any]) -> dict[str, Any] | None:
    question = first_value(row, QUESTION_KEYS)
    answer = first_value(row, ANSWER_KEYS, "")
    gold = first_value(row, GOLD_KEYS)
    if not question or not gold:
        return None
    if isinstance(gold, list):
        if len(gold) != 1:
            return None
        gold = gold[0]
    return {"question": str(question), "answer": str(answer), "doc_id": str(gold)}

docs_by_id = {}
qa_pairs = []
for row in rows:
    doc = normalize_doc(row)
    if doc:
        docs_by_id[doc["id"]] = doc
    qa = normalize_qa(row)
    if qa:
        qa_pairs.append(qa)

docs = list(docs_by_id.values())
qa_pairs = [q for q in qa_pairs if q["doc_id"] in docs_by_id]

if MAX_DOCS:
    keep_ids = {d["id"] for d in docs[:MAX_DOCS]}
    docs = [d for d in docs if d["id"] in keep_ids]
    qa_pairs = [q for q in qa_pairs if q["doc_id"] in keep_ids]
if MAX_QA:
    qa_pairs = qa_pairs[:MAX_QA]

print("docs:", len(docs))
print("qa_pairs:", len(qa_pairs))
pd.DataFrame(docs[:3])

If the automatic normalization returns zero rows because the HF schema changed, inspect the sample columns above and adjust the key lists in the previous cell.

In [ ]:
assert docs, "No documents found. Inspect dataset columns and adjust DOC_ID_KEYS/TEXT_KEYS."
assert qa_pairs, "No QA pairs found. Inspect dataset columns and adjust QUESTION_KEYS/GOLD_KEYS."
pd.DataFrame(qa_pairs[:5])

## 5. Chunk documents

In [ ]:
def split_sentences(text: str) -> list[str]:
    return [p.strip() for p in re.split(r"(?<=[.!?])\s+", text.strip()) if p.strip()]

def chunk_text(text: str, target_chars: int = CHUNK_TARGET_CHARS, overlap_sents: int = CHUNK_OVERLAP_SENTENCES) -> list[str]:
    paragraphs = [p.strip() for p in text.split("\n") if p.strip()]
    chunks = []
    current = []
    current_len = 0

    def flush():
        nonlocal current, current_len
        if current:
            chunks.append("\n".join(current))
            current = []
            current_len = 0

    for paragraph in paragraphs:
        if len(paragraph) > target_chars:
            flush()
            sents = split_sentences(paragraph)
            buf = []
            for sent in sents:
                if sum(map(len, buf)) + len(sent) > target_chars and buf:
                    chunks.append(" ".join(buf))
                    buf = buf[-overlap_sents:] if overlap_sents else []
                buf.append(sent)
            if buf:
                chunks.append(" ".join(buf))
            continue

        if current_len + len(paragraph) > target_chars and current:
            flush()
        current.append(paragraph)
        current_len += len(paragraph)
    flush()
    return chunks or [text[:target_chars]]

chunks = []
for doc in docs:
    for idx, text in enumerate(chunk_text(doc["text"])):
        chunks.append({"chunk_id": f"{doc['id']}::{idx}", "doc_id": doc["id"], "title": doc["title"], "text": text})

print("chunks:", len(chunks))
pd.DataFrame(chunks[:3])

## 6. Metrics

In [ ]:
def metrics_at_k(rankings: list[list[str]], gold_ids: list[str], ks: list[int]) -> dict[str, float]:
    out = {}
    n = len(gold_ids)
    for k in ks:
        reciprocal_ranks = []
        hits = []
        precision = []
        for ranked, gold in zip(rankings, gold_ids):
            top = ranked[:k]
            if gold in top:
                rank = top.index(gold) + 1
                reciprocal_ranks.append(1.0 / rank)
                hits.append(1.0)
                precision.append(1.0 / k)
            else:
                reciprocal_ranks.append(0.0)
                hits.append(0.0)
                precision.append(0.0)
        out[f"mrr@{k}"] = float(np.mean(reciprocal_ranks))
        out[f"recall@{k}"] = float(np.mean(hits))
        out[f"hit@{k}"] = float(np.mean(hits))
        out[f"precision@{k}"] = float(np.mean(precision))
    return out

def minmax(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    lo, hi = float(np.min(x)), float(np.max(x))
    if hi - lo < 1e-9:
        return np.zeros_like(x)
    return (x - lo) / (hi - lo)

## 7. BM25 index

In [ ]:
def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-zA-Z0-9_]+", text.lower())

chunk_texts = [c["text"] for c in chunks]
chunk_doc_ids = [c["doc_id"] for c in chunks]
title_texts = [d["title"] for d in docs]
title_doc_ids = [d["id"] for d in docs]

bm25_chunks = BM25Okapi([tokenize(t) for t in tqdm(chunk_texts, desc="tokenize chunks")])
bm25_titles = BM25Okapi([tokenize(t) for t in tqdm(title_texts, desc="tokenize titles")])

queries = [q["question"] for q in qa_pairs]
gold_doc_ids = [q["doc_id"] for q in qa_pairs]

def bm25_rank(bm25, ids: list[str], query: str, max_k: int) -> tuple[list[str], np.ndarray]:
    scores = np.asarray(bm25.get_scores(tokenize(query)), dtype=np.float32)
    order = np.argsort(-scores)[:max_k]
    return [ids[i] for i in order], scores

## 8. Dense embeddings

Colab GPU is used automatically if available.

In [ ]:
model = SentenceTransformer(EMBED_MODEL, device=device)

def encode(texts: list[str], batch_size: int = 64) -> np.ndarray:
    return model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)

chunk_vecs = encode(chunk_texts)
title_vecs = encode(title_texts)
query_vecs = encode(queries)

print("chunk_vecs:", chunk_vecs.shape)
print("title_vecs:", title_vecs.shape)
print("query_vecs:", query_vecs.shape)

## 9. Evaluate BM25, vector and hybrid

In [ ]:
MAX_K = max(EVAL_KS)

def dense_rank(matrix: np.ndarray, ids: list[str], qvec: np.ndarray, max_k: int) -> tuple[list[str], np.ndarray]:
    scores = matrix @ qvec
    order = np.argsort(-scores)[:max_k]
    return [ids[i] for i in order], scores

def hybrid_rank(bm25_scores: np.ndarray, dense_scores: np.ndarray, ids: list[str], alpha: float, max_k: int) -> list[str]:
    score = alpha * minmax(dense_scores) + (1 - alpha) * minmax(bm25_scores)
    order = np.argsort(-score)[:max_k]
    return [ids[i] for i in order]

def evaluate_chunks() -> list[dict[str, Any]]:
    rows_out = []
    bm25_rankings = []
    vector_rankings = []
    hybrid_rankings_by_alpha = {a: [] for a in ALPHAS}

    for q, qv in tqdm(list(zip(queries, query_vecs)), desc="eval chunks"):
        bm25_ids, bm25_scores = bm25_rank(bm25_chunks, chunk_doc_ids, q, MAX_K)
        vector_ids, dense_scores = dense_rank(chunk_vecs, chunk_doc_ids, qv, MAX_K)
        bm25_rankings.append(bm25_ids)
        vector_rankings.append(vector_ids)
        for alpha in ALPHAS:
            hybrid_rankings_by_alpha[alpha].append(hybrid_rank(bm25_scores, dense_scores, chunk_doc_ids, alpha, MAX_K))

    rows_out.append({"task": "chunks", "method": "bm25", "alpha": None, **metrics_at_k(bm25_rankings, gold_doc_ids, EVAL_KS)})
    rows_out.append({"task": "chunks", "method": "vector", "alpha": None, **metrics_at_k(vector_rankings, gold_doc_ids, EVAL_KS)})
    for alpha, rankings in hybrid_rankings_by_alpha.items():
        rows_out.append({"task": "chunks", "method": "hybrid_linear", "alpha": alpha, **metrics_at_k(rankings, gold_doc_ids, EVAL_KS)})
    return rows_out

def evaluate_titles() -> list[dict[str, Any]]:
    rows_out = []
    bm25_rankings = []
    vector_rankings = []
    hybrid_rankings_by_alpha = {a: [] for a in ALPHAS}

    for q, qv in tqdm(list(zip(queries, query_vecs)), desc="eval titles"):
        bm25_ids, bm25_scores = bm25_rank(bm25_titles, title_doc_ids, q, MAX_K)
        vector_ids, dense_scores = dense_rank(title_vecs, title_doc_ids, qv, MAX_K)
        bm25_rankings.append(bm25_ids)
        vector_rankings.append(vector_ids)
        for alpha in ALPHAS:
            hybrid_rankings_by_alpha[alpha].append(hybrid_rank(bm25_scores, dense_scores, title_doc_ids, alpha, MAX_K))

    rows_out.append({"task": "titles", "method": "bm25", "alpha": None, **metrics_at_k(bm25_rankings, gold_doc_ids, EVAL_KS)})
    rows_out.append({"task": "titles", "method": "vector", "alpha": None, **metrics_at_k(vector_rankings, gold_doc_ids, EVAL_KS)})
    for alpha, rankings in hybrid_rankings_by_alpha.items():
        rows_out.append({"task": "titles", "method": "hybrid_linear", "alpha": alpha, **metrics_at_k(rankings, gold_doc_ids, EVAL_KS)})
    return rows_out

results = pd.DataFrame(evaluate_chunks() + evaluate_titles())
results.sort_values(["task", "mrr@10"], ascending=[True, False]).head(20)

## 10. Summaries and top-k recommendation

In [ ]:
def best_by_task_metric(df: pd.DataFrame, task: str, metric: str) -> pd.DataFrame:
    return df[df["task"] == task].sort_values(metric, ascending=False).head(10)

display(best_by_task_metric(results, "chunks", "mrr@10"))
display(best_by_task_metric(results, "titles", "hit@10"))

In [ ]:
def pick_top_k(row: pd.Series, metric: str, ks: list[int], target_ratio: float = 0.95, max_k: int | None = None):
    ks2 = [k for k in ks if max_k is None or k <= max_k]
    values = [(k, float(row[f"{metric}@{k}"])) for k in ks2]
    k_max, v_max = values[-1]
    threshold = target_ratio * v_max
    for k, v in values:
        if v >= threshold:
            return k, v, v_max
    return k_max, v_max, v_max

best_chunks = best_by_task_metric(results, "chunks", "mrr@10").iloc[0]
best_titles = best_by_task_metric(results, "titles", "hit@10").iloc[0]

chunk_k, chunk_v, chunk_max = pick_top_k(best_chunks, "recall", EVAL_KS, target_ratio=0.95, max_k=15)
title_k, title_v, title_max = pick_top_k(best_titles, "hit", EVAL_KS, target_ratio=0.95, max_k=10)

summary = pd.DataFrame([
    {
        "tool": "search_by_chunks",
        "method": best_chunks["method"],
        "alpha": best_chunks["alpha"],
        "primary_metric": "mrr@10",
        "primary_value": best_chunks["mrr@10"],
        "recommended_top_k_by_95_recall": chunk_k,
        "note": "Use smaller k in production if context cost is important.",
    },
    {
        "tool": "search_by_titles",
        "method": best_titles["method"],
        "alpha": best_titles["alpha"],
        "primary_metric": "hit@10",
        "primary_value": best_titles["hit@10"],
        "recommended_top_k_by_95_hit": title_k,
        "note": "Title search is document-level; k=10 is usually enough.",
    },
])
display(summary)

In [ ]:
results.to_csv("retrieval_eval_results.csv", index=False)
summary.to_csv("retrieval_eval_summary.csv", index=False)
print("Saved retrieval_eval_results.csv and retrieval_eval_summary.csv")

## Interpretation template

Use this template after the run:

1. Compare BM25, vector and hybrid by `MRR@10` for chunks.
2. Compare BM25 title, vector title and hybrid title by `Hit@10`.
3. Select `alpha` by the best primary metric.
4. Select `top_k` by the metric curve and context budget.
5. Document the trade-off: higher `top_k` improves recall but increases prompt size and noise.

The local repository currently uses `embeddinggemma`, `alpha=0.7`, `top_k=8` for chunks and `alpha=1.0`, `top_k=10` for titles.